In [1]:
import math
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V):
    # Extract the dimensionality of the key vectors
    d_k = Q.size(-1)
    
    # 1. Compute raw alignment scores via matrix multiplication (Q @ K^T)
    # K.transpose(-2, -1) cleanly flips the last two dimensions of the Key matrix
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # 2. Scale scores to prevent vanishing gradients during softmax
    scaled_scores = scores / math.sqrt(d_k)
    
    # 3. Apply Softmax to generate the attention weight distribution matrix
    attention_weights = F.softmax(scaled_scores, dim=-1)
    
    # 4. Multiply weights by Value vectors to get the final contextual representation
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# --- Verify Behavior with a Mock Text Sequence ---
# Batch size = 1, Sequence length = 4 tokens, Vector dimension = 8
torch.manual_seed(42)
mock_Q = torch.randn(1, 4, 8)
mock_K = torch.randn(1, 4, 8)
mock_V = torch.randn(1, 4, 8)

context_tensors, weights = scaled_dot_product_attention(mock_Q, mock_K, mock_V)

print("--- Self-Attention Metrics ---")
print(f"Output Context Tensor Shape: {context_tensors.shape} (Batch, Seq_Len, Dim)")
print(f"Attention Matrix Shape:      {weights.shape} (Batch, Seq_Len, Seq_Len)\n")

print("--- Visualizing the Attention Matrix ---")
# Each row represents a token looking at every other token in the sequence
print(weights[0].round(decimals=4))

--- Self-Attention Metrics ---
Output Context Tensor Shape: torch.Size([1, 4, 8]) (Batch, Seq_Len, Dim)
Attention Matrix Shape:      torch.Size([1, 4, 4]) (Batch, Seq_Len, Seq_Len)

--- Visualizing the Attention Matrix ---
tensor([[0.1942, 0.4102, 0.3242, 0.0714],
        [0.0408, 0.6555, 0.0864, 0.2173],
        [0.1041, 0.2057, 0.1824, 0.5078],
        [0.1926, 0.1803, 0.1955, 0.4316]])
